# ExDark → Zero-DCE++ → Bilateral → YOLOv8n

Zero-DCE++ uses the **official pretrained weights** through the code in `src/hienanh`; Zero-DCE++ is not retrained. YOLOv8n is trained on images that have undergone both processing stages. The final metrics must come from your actual run.


## Block 1 — Check the GPU and define common parameters


In [ ]:
from pathlib import Path
import torch

PROJECT = Path('/kaggle/working/Project18')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME = 'yolov8n.pt'
EPOCHS = 40
IMAGE_SIZE = 640
BATCH_SIZE = 16
PATIENCE = 15
SEED = 42
print('PyTorch:', torch.__version__)
print('Device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
assert DEVICE == 'cuda', 'Enable the GPU in Settings → Accelerator, then run this block again.'


## Block 2 — Get the project from GitHub
If cloning reports `Could not resolve host`, enable Internet access in Kaggle Settings. If the branch cannot be found, first push the `feature/denoised-dataset` branch from your local machine.


In [ ]:
import subprocess

REPO_URL = 'https://github.com/NMH1203/Low-Light-Image-Enhancement-and-Downstream-Recognition.git'
BRANCH = 'feature/denoised-dataset'
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(PROJECT)], check=True)
else:
    print('The project already exists; using the existing copy:', PROJECT)
assert (PROJECT / 'train_ninh.py').is_file(), 'Missing train_ninh.py'
assert (PROJECT / 'src/hienanh/enhancement.py').is_file(), 'Missing the Hien Anh module'
assert (PROJECT / 'src/create_denoised_dataset.py').is_file(), 'Missing the Bilateral script; check the pushed branch.'
subprocess.run(['git', 'branch', '--show-current'], cwd=PROJECT, check=True)


## Block 3 — Install dependencies and check the dark-image dataset


In [ ]:
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT / 'requirements.txt')], check=True)
import cv2, yaml, ultralytics
print('OpenCV:', cv2.__version__, '| Ultralytics:', ultralytics.__version__)
DARK = PROJECT / 'Dataset/exdark_yolo_dark'
expected = {'train': 5142, 'valid': 1469, 'test': 734}
for split, count in expected.items():
    images = [p for p in (DARK / split / 'images').iterdir() if p.is_file()]
    labels = list((DARK / split / 'labels').glob('*.txt'))
    print(split, 'images:', len(images), 'labels:', len(labels))
    assert len(images) == len(labels) == count, f'{split}: incorrect image/label count'


## Block 4 — Standardize labels and check for leakage across splits
`train` is used for learning, `valid` for model selection, and `test` only for the final evaluation. This step standardizes the labels; it does not train the model.


In [ ]:
CLEAN = PROJECT / 'Dataset/exdark_yolo_clean'
if not CLEAN.exists():
    subprocess.run([sys.executable, 'train_ninh.py', 'prepare', '--source', str(DARK), '--output', str(CLEAN), '--seed', str(SEED)], cwd=PROJECT, check=True)
else:
    print('The clean dataset already exists; validating it before use.')
assert (CLEAN / 'data.yaml').is_file()
for split, count in expected.items():
    n_img = len(list((CLEAN / split / 'images').iterdir()))
    n_lab = len(list((CLEAN / split / 'labels').glob('*.txt')))
    print(split, n_img, n_lab)
    assert n_img == n_lab == count


## Block 5 — Find or download the official Zero-DCE++ weights
`Epoch99.pth` contains the weights published by the Zero-DCE++ authors. This block checks the SHA-256 file fingerprint to prevent use of the wrong checkpoint.


In [ ]:
import hashlib
WEIGHTS = PROJECT / 'src/hienanh/Results/weights/zerodcepp_Epoch99.pth'
EXPECTED_SHA256 = 'ca8855b90df9a80fa4195a831f33d3476b1964f787eb70602797c773067f3b84'
if not WEIGHTS.is_file():
    print('The weights are not available in Kaggle yet; downloading them with the Hien Anh script...')
    subprocess.run([sys.executable, 'src/hienanh/download_weights.py'], cwd=PROJECT, check=True)
assert WEIGHTS.is_file(), f'Could not find {WEIGHTS}'
sha = hashlib.sha256()
with WEIGHTS.open('rb') as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b''):
        sha.update(chunk)
assert sha.hexdigest() == EXPECTED_SHA256, 'Checksum mismatch: the checkpoint is corrupted or is not the official version.'
print('Valid weights:', WEIGHTS, '| SHA-256:', sha.hexdigest())


## Block 6 — Load Zero-DCE++ and test one image
A scale factor of 4 is the configuration used in Hien Anh's implementation. `training=False` means that the model is performing inference and is not updating its weights.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, str(PROJECT))
from src.hienanh.enhancement import load_model, enhance_neural
zero_model = load_model(WEIGHTS, variant='zerodcepp', scale_factor=4, device=DEVICE)
print('Number of parameters:', sum(p.numel() for p in zero_model.parameters()), '| training:', zero_model.training)
sample = sorted((CLEAN / 'test/images').iterdir())[0]
raw = cv2.imdecode(np.fromfile(sample, dtype=np.uint8), cv2.IMREAD_COLOR)
assert raw is not None
bright = enhance_neural(raw, zero_model, device=DEVICE)
assert bright.shape == raw.shape
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, image, title in zip(axes, [raw, bright], ['Dark', 'Zero-DCE++']):
    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)); ax.set_title(title); ax.axis('off')
plt.show()
print('Test image:', sample.name, '| Shape:', raw.shape)


## Block 7 — Enhance all train/valid/test images
This block creates PNG images with unchanged dimensions and copies their labels. If the Kaggle session is interrupted, run the block again to resume. The output may require several GB of storage.


In [ ]:
import json, shutil
ZERO = PROJECT / 'Dataset/exdark_yolo_zerodcepp'
extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
for split, target_count in expected.items():
    in_images, in_labels = CLEAN / split / 'images', CLEAN / split / 'labels'
    out_images, out_labels = ZERO / split / 'images', ZERO / split / 'labels'
    out_images.mkdir(parents=True, exist_ok=True)
    out_labels.mkdir(parents=True, exist_ok=True)
    files = sorted(p for p in in_images.iterdir() if p.suffix.lower() in extensions)
    assert len(files) == target_count
    for index, image_path in enumerate(files, 1):
        label = in_labels / (image_path.stem + '.txt')
        dest = out_images / (image_path.stem + '.png')
        out_label = out_labels / label.name
        assert label.is_file(), f'Missing label: {label}'
        if not dest.is_file() or not out_label.is_file():
            original = cv2.imdecode(np.fromfile(image_path, dtype=np.uint8), cv2.IMREAD_COLOR)
            assert original is not None, f'Could not read {image_path}'
            result = enhance_neural(original, zero_model, device=DEVICE)
            assert result.shape == original.shape
            ok, encoded = cv2.imencode('.png', result, [cv2.IMWRITE_PNG_COMPRESSION, 3])
            assert ok
            temporary = dest.with_suffix('.png.tmp')
            encoded.tofile(temporary)
            temporary.replace(dest)
            shutil.copy2(label, out_label)
        if index % 250 == 0 or index == len(files):
            print(f'{split}: {index}/{len(files)}', flush=True)
with (CLEAN / 'data.yaml').open(encoding='utf-8-sig') as stream:
    source_config = yaml.safe_load(stream)
zero_config = {'path': str(ZERO.resolve()), 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images', 'names': source_config['names']}
(ZERO / 'data.yaml').write_text(yaml.safe_dump(zero_config, sort_keys=False, allow_unicode=True), encoding='utf-8')
metadata = {'method': 'official pretrained Zero-DCE++', 'variant': 'zerodcepp', 'scale_factor': 4, 'checkpoint': str(WEIGHTS), 'checkpoint_sha256': EXPECTED_SHA256, 'source': str(CLEAN), 'train_on_exdark': False, 'counts': expected}
(ZERO / 'enhancement_metadata.json').write_text(json.dumps(metadata, indent=2) + '\n', encoding='utf-8')
print('Zero-DCE++ dataset:', ZERO)


## Block 8 — Validate the Zero-DCE++ dataset
Check that image and label filenames correspond instead of checking only their total counts.


In [ ]:
for split, count in expected.items():
    image_stems = {p.stem for p in (ZERO / split / 'images').glob('*.png')}
    label_stems = {p.stem for p in (ZERO / split / 'labels').glob('*.txt')}
    print(split, 'images:', len(image_stems), 'labels:', len(label_stems))
    assert image_stems == label_stems and len(image_stems) == count
print('The Zero-DCE++ dataset is complete.')


## Block 9 — Test the Bilateral filter on five images from each split
The Bilateral filter reduces noise based on pixel distance and color similarity, and usually preserves edges better than conventional blurring filters.


In [ ]:
SMOKE = PROJECT / 'Dataset/exdark_yolo_zerodcepp_bilateral_smoke'
if not SMOKE.exists():
    subprocess.run([sys.executable, 'src/create_denoised_dataset.py', '--source', str(ZERO), '--output', str(SMOKE), '--diameter', '5', '--sigma-color', '25', '--sigma-space', '25', '--workers', '4', '--limit', '5'], cwd=PROJECT, check=True)
else:
    print('The smoke-test dataset already exists:', SMOKE)
for split in expected:
    assert len(list((SMOKE / split / 'images').iterdir())) == 5
print('The Bilateral smoke test passed.')


## Block 10 — Compare three versions of the same source image
Inspect noise, small-object edges, illuminated regions, and colors. If filtering blurs objects, reduce `sigma` or `diameter` before processing the full dataset.


In [ ]:
example = sorted((CLEAN / 'test/images').iterdir())[0]
paths = [example, ZERO / 'test/images' / (example.stem + '.png'), SMOKE / 'test/images' / (example.stem + '.png')]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, path, title in zip(axes, paths, ['Dark', 'Zero-DCE++', 'Zero-DCE++ + Bilateral']):
    image = cv2.imread(str(path))
    assert image is not None, path
    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)); ax.set_title(title); ax.axis('off')
plt.show()
print('Comparison image:', example.name)


## Block 11 — Create the complete Bilateral-filtered dataset
This dataset is used as input to YOLO. Keep `d=5`, `sigmaColor=25`, and `sigmaSpace=25` for comparisons, and report these exact values.


In [ ]:
DENOISED = PROJECT / 'Dataset/exdark_yolo_zerodcepp_bilateral_d5_s25'
if not DENOISED.exists():
    subprocess.run([sys.executable, 'src/create_denoised_dataset.py', '--source', str(ZERO), '--output', str(DENOISED), '--diameter', '5', '--sigma-color', '25', '--sigma-space', '25', '--workers', '4'], cwd=PROJECT, check=True)
else:
    print('The Bilateral-filtered dataset already exists; checking that it is complete.')
assert (DENOISED / 'data.yaml').is_file()
for split, count in expected.items():
    image_stems = {p.stem for p in (DENOISED / split / 'images').iterdir() if p.is_file()}
    label_stems = {p.stem for p in (DENOISED / split / 'labels').glob('*.txt')}
    print(split, len(image_stems), len(label_stems))
    assert image_stems == label_stems and len(image_stems) == count
print('Dataset ready for YOLO:', DENOISED / 'data.yaml')


## Block 12 — Smoke-test YOLOv8n training for two epochs
The smoke test checks `data.yaml`, labels, the GPU, and memory usage. If VRAM is insufficient, set `BATCH_SIZE = 8` in Block 1 and rerun from Block 12.


In [ ]:
RESULTS = PROJECT / 'Results/tiep'
smoke_run = RESULTS / 'yolo_zerodcepp_bilateral_smoke'
assert MODEL_NAME == 'yolov8n.pt', 'train_ninh.py currently fixes the model to yolov8n.pt'
if not smoke_run.exists():
    subprocess.run([sys.executable, 'train_ninh.py', 'train', '--data', str(DENOISED / 'data.yaml'), '--project', str(RESULTS), '--name', smoke_run.name, '--epochs', '2', '--batch', str(BATCH_SIZE), '--imgsz', str(IMAGE_SIZE), '--device', '0', '--hsv-v', '0.1', '--close-mosaic', '2', '--patience', '2', '--seed', str(SEED)], cwd=PROJECT, check=True)
else:
    print('The smoke-test run already exists:', smoke_run)
assert (smoke_run / 'weights/best.pt').is_file(), 'The smoke test is incomplete; inspect the logs and use a new run name if necessary.'


## Block 13 — Run the full YOLOv8n training
Parameters: a maximum of 40 epochs, image size 640, batch size 16, patience 15, and seed 42. `best.pt` is the best checkpoint according to validation performance.


In [ ]:
YOLO_RUN = RESULTS / 'yolo_zerodcepp_bilateral_e40'
if not YOLO_RUN.exists():
    subprocess.run([sys.executable, 'train_ninh.py', 'train', '--data', str(DENOISED / 'data.yaml'), '--project', str(RESULTS), '--name', YOLO_RUN.name, '--epochs', str(EPOCHS), '--batch', str(BATCH_SIZE), '--imgsz', str(IMAGE_SIZE), '--device', '0', '--hsv-v', '0.1', '--close-mosaic', '10', '--patience', str(PATIENCE), '--seed', str(SEED)], cwd=PROJECT, check=True)
else:
    print('The YOLO run already exists:', YOLO_RUN)
BEST = YOLO_RUN / 'weights/best.pt'
assert BEST.is_file(), 'The YOLO run does not contain best.pt; inspect the logs. The script does not currently resume automatically.'
print('Checkpoint YOLO:', BEST)


## Block 14 — Check the YOLO parameters used


In [ ]:
args = yaml.safe_load((YOLO_RUN / 'args.yaml').read_text())
for key in ['model', 'epochs', 'imgsz', 'batch', 'patience', 'seed', 'hsv_v', 'close_mosaic']:
    print(key, ':', args.get(key))
assert args['epochs'] == EPOCHS and args['imgsz'] == IMAGE_SIZE and args['batch'] == BATCH_SIZE
assert args['patience'] == PATIENCE and args['seed'] == SEED


## Block 15 — Evaluate on the test split exactly once
Precision is the proportion of predictions that are correct; recall is the proportion of actual objects that are detected; mAP50-95 evaluates bounding boxes across multiple matching thresholds and is more stringent than mAP50.


In [ ]:
TEST_RUN = RESULTS / 'test_zerodcepp_bilateral'
metrics_file = TEST_RUN / 'measured_metrics.json'
if not metrics_file.is_file():
    assert not TEST_RUN.exists(), 'An incomplete evaluation directory already exists; inspect it, then use a new run name.'
    subprocess.run([sys.executable, 'train_ninh.py', 'evaluate', '--data', str(DENOISED / 'data.yaml'), '--weights', str(BEST), '--project', str(RESULTS), '--name', TEST_RUN.name, '--split', 'test', '--batch', str(BATCH_SIZE), '--imgsz', str(IMAGE_SIZE), '--device', '0'], cwd=PROJECT, check=True)
metrics = json.loads(metrics_file.read_text())
print(json.dumps(metrics, indent=2))


## Block 16 — Export the measured results table
This table contains only one row for your experiment. Combine it with the team's Raw and Zero-DCE++ results to determine whether the Bilateral filter improves mAP.


In [ ]:
import pandas as pd
table = pd.DataFrame([{'experiment': 'Zero-DCE++ + Bilateral + YOLOv8n', 'enhancer': 'official pretrained Zero-DCE++', 'bilateral_d': 5, 'sigma_color': 25, 'sigma_space': 25, 'epochs_max': EPOCHS, 'imgsz': IMAGE_SIZE, 'batch': BATCH_SIZE, 'patience': PATIENCE, 'seed': SEED, 'precision': metrics['precision'], 'recall': metrics['recall'], 'map50': metrics['map50'], 'map50_95': metrics['map50_95']}])
summary_file = RESULTS / 'dcepp_bilateral_summary.csv'
table.to_csv(summary_file, index=False)
display(table)
print('Table:', summary_file)


## Block 17 — View plots and the confusion matrix


In [ ]:
from IPython.display import Image, display
for path in [YOLO_RUN / 'results.png', TEST_RUN / 'confusion_matrix.png', TEST_RUN / 'PR_curve.png', TEST_RUN / 'F1_curve.png']:
    print(path.name, '→', path.exists())
    if path.exists():
        display(Image(filename=str(path)))


## Block 18 — Export images with YOLO predictions
`conf=0.25` affects only the rendered example images; it does not change the mAP calculated in Block 15.


In [ ]:
PRED_RUN = RESULTS / 'pred_zerodcepp_bilateral'
if not PRED_RUN.exists():
    subprocess.run(['yolo', 'detect', 'predict', f'model={BEST}', f'source={DENOISED / "test/images"}', f'imgsz={IMAGE_SIZE}', 'conf=0.25', 'device=0', 'save=True', 'save_txt=True', 'save_conf=True', f'project={RESULTS}', f'name={PRED_RUN.name}'], cwd=PROJECT, check=True)
images = sorted(p for p in PRED_RUN.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
print('Number of prediction images:', len(images))
assert images, 'No prediction images were generated'
for image_path in images[:6]:
    display(Image(filename=str(image_path)))


## Block 19 — Check the outputs and download the results
The ZIP file contains the checkpoint, metrics, plots, and prediction images. The processed image datasets are stored separately in `/kaggle/working/Project18/Dataset` and are excluded from this ZIP to avoid making it too large. Save a Kaggle version or download the ZIP before ending the session.


In [ ]:
from IPython.display import FileLink
required = [WEIGHTS, ZERO / 'enhancement_metadata.json', DENOISED / 'denoising_metadata.json', DENOISED / 'data.yaml', BEST, YOLO_RUN / 'args.yaml', YOLO_RUN / 'results.csv', metrics_file, summary_file]
for path in required:
    print('OK' if path.is_file() else 'MISSING', path)
assert all(path.is_file() for path in required)
archive = shutil.make_archive('/kaggle/working/tiep_dcepp_bilateral_results', 'zip', root_dir=RESULTS)
print('Created:', archive)
display(FileLink(archive))
